In [2]:
#import necessary package
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

In [3]:
# Build Fruits classification using MobileNetV2

transform = transforms.Compose([
    transforms.Resize((224, 224)),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
print("Fruit labels:", class_names)

Fruit labels: ['Apple', 'Cherry', 'Tomatoe']


In [4]:
# Load pretrained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
for param in model.features.parameters():
    param.requires_grad = False
# Replace classifier head
num_classes = len(class_names)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.last_channel, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.3),
    nn.Linear(128, num_classes),
    nn.Softmax(dim=1)
)

In [ ]:
# Check availability of GPU
print(torch.cuda.is_available())   # should print True
print(torch.cuda.get_device_name(0))  # e.g. 'Tesla T4'

In [6]:
# Training SetUp
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [7]:
# Training Loop
for epoch in range(5):  # adjust epochs
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Train Acc: {100*correct/total:.2f}%")

#  Validation
model.eval()
val_correct, val_total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100*val_correct/val_total:.2f}%")

# Save model back to Drive
torch.save(model, "fruit_mobilenetv2.pth")


Epoch 1, Loss: 0.6104, Train Acc: 94.61%
Epoch 2, Loss: 0.5717, Train Acc: 98.15%
Epoch 3, Loss: 0.5662, Train Acc: 98.62%
Epoch 4, Loss: 0.5633, Train Acc: 98.87%
Epoch 5, Loss: 0.5607, Train Acc: 99.12%
Validation Accuracy: 89.35%
